# NanoSAM image encoder 蒸餾訓練（Colab 版）

把 SAM 的影像編碼器蒸餾成 ResNet18（MobileSAM 當老師）。**由上往下一格一格執行，或直接『執行階段 → 全部執行』即可。**

**事前準備（只做一次）**
1. 執行階段 → 變更執行階段類型 → **GPU**（T4 即可）
2. 在 Google Drive 建立資料夾 `MyDrive/nanosam_train/`，放入：
   - `nanosam_train_bundle.tar.gz`（主機執行 `./training/remote/make_bundle.sh` 產生）
   - `coco/val2017.zip`（COCO val2017 影像，778 MB，你手動上傳；**用 zip 最快**，不要傳 5000 個散檔）
   - （選用）要接續主機已訓練的進度：把主機的 `data/train/resnet18_run1/` 整個放到 `backup/resnet18_run1/`

**運作方式**
- 所有東西（環境、COCO、訓練）都在 Colab 本機磁碟 `/content` 進行，速度快
- **每個 epoch 結束**，`checkpoint.pth / best.pth / log.txt` 自動備份到 Drive 的 `nanosam_train/backup/resnet18_run1/`
- **斷線 / 換新的執行階段**：重新『全部執行』即可，會自動從 Drive 備份還原並接續
- 想改訓練長度：只改第 2 格的 `EPOCHS`（= 總共跑到第幾個 epoch）
- 要和主機結果可比較：保持 `BATCH × ACCUM = 16`、`VAL = 200`（預設）

In [ ]:
# 1) 確認有 GPU
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
!python3 --version
!df -h /content | tail -1

In [ ]:
# 2) 設定 + 掛載 Google Drive（要改的只有這一格）
EPOCHS = 10                 # 總共訓練到第幾個 epoch（主機已跑 3 個；訓練會自動接續）
RUN = 'resnet18_run1'

from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE_DIR  = '/content/drive/MyDrive/nanosam_train'
BUNDLE     = DRIVE_DIR + '/nanosam_train_bundle.tar.gz'
COCO_SRC   = DRIVE_DIR + '/coco/val2017.zip'
BACKUP_DIR = DRIVE_DIR + '/backup'
os.makedirs(BACKUP_DIR, exist_ok=True)
for name, p in [('訓練包', BUNDLE), ('COCO zip', COCO_SRC)]:
    assert os.path.exists(p), f'找不到{name}：{p}\n請依上方「事前準備」放到 Drive'
    print(f'OK  {name}: {p}  ({os.path.getsize(p)/2**20:.0f} MB)')
ck = f'{BACKUP_DIR}/{RUN}/checkpoint.pth'
print('備份中已有 checkpoint → 會接續訓練' if os.path.exists(ck) else '備份中沒有 checkpoint → 從頭訓練')
os.environ.update(SYSTEM_PYTHON='1', EPOCHS=str(EPOCHS), RUN=RUN, COCO_SRC=COCO_SRC, BACKUP_DIR=BACKUP_DIR)

In [ ]:
# 3) 解開訓練包到 /content，建環境（PyTorch 2.5.1 + nanosam），下載並驗證 baseline 權重（約 3~6 分鐘）
!tar xzf "{BUNDLE}" -C /content
%cd /content/nanosam_train
!cat VERSION
!./run_remote.sh setup

In [ ]:
# 4) 把 Drive 上的 COCO zip 複製到本機 /content 並解壓（之後訓練都讀本機檔案）
!./run_remote.sh data

In [ ]:
# 5) 小測：估算一個 epoch 要多久（約 1 分鐘；可略過）
!./run_remote.sh smoke

In [ ]:
# 6) 訓練（前景執行，一直跑到 EPOCHS 為止）
#    每個 epoch 結束會印一行 [備份] 並把檔案存到 Drive；中途斷線就重新『全部執行』即可接續
!./run_remote.sh train

In [ ]:
# 7) 看結果與備份（每個 epoch 一行；*best = 驗證最佳）
!cat data/train/{RUN}/log.txt
!ls -lh "{BACKUP_DIR}/{RUN}/"

**訓練完成後**：`best.pth`（約 60 MB）已在 Drive 的 `nanosam_train/backup/resnet18_run1/`。下載後放到主機的 `data/train/resnet18_run1/`，再執行：

```bash
python training/compare_encoders.py --student data/train/resnet18_run1/best.pth
```

（延遲請在主機上量，baseline 與自訓練必須在同一台機器上比較。）